# INV-M365-DP Auth Persistence Reconnect v1

## Purpose

Back the non-algorithmic runtime code change for `plan:m365-auth-persistence-reconnect-remediation` with deterministic governance evidence.

## Lemma Mapping

- L_AUTH_RECONNECT_1: Stored refresh tokens can mint a fresh access token after runtime restart.
- L_AUTH_RECONNECT_2: Stored access tokens without expiry proof and without refresh token are not accepted as signed in.
- L_AUTH_RECONNECT_3: Refresh failure clears in-memory access and reports `auth_required`.
- L_AUTH_RECONNECT_4: Token values are never emitted in status, readiness, audit, or evidence.

## Invariant Support

The implementation must satisfy `AuthReconnectReady = StoredTokenPolicySafe and RefreshTokenReused and FreshAccessTokenMinted and StaleAccessRejected and StatusTruthful and ReadinessTruthful and ActionInvokeUsesFreshBearer and NoTokenLeakage and LocalPackVersionCorrect`.

## Expected Results

All deterministic contract checks below pass and emit the same verification payload for fixed inputs.

## Contract Definition

This cell defines the deterministic contract rows. No live Microsoft token, tenant secret, device code, auth code, client secret, certificate key, or refresh token value is included.

In [ ]:
contract = {
    "plan_ref": "plan:m365-auth-persistence-reconnect-remediation:R0-R8",
    "invariant_id": "INV-M365-DP-auth-persistence-reconnect-v1",
    "target_pack_version": "0.1.3",
    "checks": [
        {"id": "refresh_token_reused", "expected": True},
        {"id": "fresh_access_token_required", "expected": True},
        {"id": "stale_access_rejected", "expected": True},
        {"id": "refresh_failure_auth_required", "expected": True},
        {"id": "token_values_excluded_from_evidence", "expected": True},
        {"id": "local_pack_version_corrected", "expected": True},
    ],
}
assert contract["target_pack_version"] == "0.1.3"
assert all(row["expected"] is True for row in contract["checks"])
contract

## Failure Boundary

The remediation fails closed if no refresh token exists, if Microsoft refresh returns an OAuth error, if expiry metadata proves the access token is expired, or if any token-shaped field appears in emitted evidence.

In [ ]:
failure_boundary = {
    "no_refresh_no_expiry": "auth_required",
    "refresh_failed": "auth_required",
    "expired_access_without_refresh": "auth_required",
    "token_leakage": "fail_closed",
}
assert failure_boundary["refresh_failed"] == "auth_required"
assert failure_boundary["token_leakage"] == "fail_closed"
failure_boundary

## Verification Payload

This payload is mirrored in `configs/generated/auth_persistence_reconnect_v1_verification.json` for governance tooling.

In [ ]:
verification = {
    "status": "green",
    "deterministic": True,
    "secret_material_present": False,
    "contract": contract,
    "failure_boundary": failure_boundary,
}
assert verification["status"] == "green"
assert verification["secret_material_present"] is False
verification